In [1]:
import os
import torch
from torch.utils.data import DataLoader
from dataset.dataloader import BrainDataset_light
import tqdm
import numpy as np
base_path ='/Users/yifanli/Desktop/dataset/TumorSegmentation/BraTS20/MICCAI_BraTS2020_TrainingData'

# Initialize lists for each modality and labels
flair_paths = []
t1_paths = []
t1gd_paths = []
t2_paths = []
label_paths = []

# Initialize a dictionary to track label availability
label_availability = {}

# Loop through each patient folder
for patient_folder in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_folder)
    
    if os.path.isdir(patient_path):
        # Reset or declare label found flag for each patient
        label_found = False
        
        for file in os.listdir(patient_path):
            file_path = os.path.join(patient_path, file)

            if file.endswith('_flair.nii.gz'):
                flair_paths.append(file_path)
            elif file.endswith('_t1.nii.gz'):
                t1_paths.append(file_path)
            elif file.endswith('_t1ce.nii.gz'):
                t1gd_paths.append(file_path)
            elif file.endswith('_t2.nii.gz'):
                t2_paths.append(file_path)
            elif file.endswith('_seg.nii.gz'):
                label_paths.append(file_path)
                label_found = True
                label_availability[patient_folder] = 'ManuallyCorrected'

        # If manually corrected label is not found, look for GlistrBoost label
        if not label_found:
            for file in os.listdir(patient_path):
                if file.endswith('_GlistrBoost.nii.gz'):
                    print(patient_path)
                    file_path = os.path.join(patient_path, file)
                    label_paths.append(file_path)
                    label_availability[patient_folder] = 'GlistrBoost'

# Print the count of files in each list and availability of labels
print(f'FLAIR files: {len(flair_paths)}')
print(f'T1 files: {len(t1_paths)}')
print(f'T1Gd files: {len(t1gd_paths)}')
print(f'T2 files: {len(t2_paths)}')
print(f'Label files: {len(label_paths)}')

# Optionally, print the type of label used for each patient
# for patient, label_type in label_availability.items():
#     print(f'{patient}: {label_type}')

FLAIR files: 369
T1 files: 369
T1Gd files: 369
T2 files: 369
Label files: 369


In [2]:
from models.CKD_model import  CKD_pathes
model =  CKD_pathes(embed_dim=32, output_dim=3, img_size=(128, 128, 128), patch_size=(4, 4, 4), in_chans=1, depths=[2, 2, 2], num_heads=[2, 4, 8, 16], window_size=(7, 7, 7), mlp_ratio=4.)
model.load_state_dict(torch.load("/Users/yifanli/Desktop/P3SOTA/CKD-TransBTS-main/best_model_BraTs.pkl", map_location='cpu'))
model.eval()
print()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda = True if torch.cuda.is_available() else False
if cuda:
    model.cuda()

/opt/anaconda3/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:3484.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [3]:
train_dataset = BrainDataset_light(
    t1_paths=t1_paths,
    t1gd_paths=t1gd_paths,
    t2_paths=t2_paths,
    flair_paths=flair_paths,
    label_paths=label_paths,
    is_train=True
)
data_loader = DataLoader(train_dataset, batch_size=2, shuffle=False)

In [4]:
for data in data_loader:
    patient_image = np.stack([
        data['img']['t1'],
        data['img']['t1gd'],
        data['img']['t2'],
        data['img']['flair_data']
    ], axis=1)
    patient_image = torch.from_numpy(np.array(patient_image, dtype=np.float32)).to(device)
    with torch.no_grad():
        # Get embeddings from model
         (t1_1, t1ce_1, t2_1, flair_1),(t1_2, t1ce_2, t2_2, flair_2) = model(patient_image)
    break

In [5]:
print(t1_1.shape, t1_2.shape)

torch.Size([2, 16, 64, 64, 64]) torch.Size([2, 32, 32, 32, 32])
